In [1]:
# Importar

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import mlflow
import mlflow.tensorflow
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
# Cargar datos
df = pd.read_csv("df_encoded.csv")

# Separar características y etiqueta
X = df.drop("puntaje_cat", axis=1)
y = df["puntaje_cat"]

# División de datos
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=100)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=100)

In [ ]:
# Convertir a tf.data.Dataset
def df_to_dataset(X, y, shuffle=True, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((dict(X), y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X))
    return ds.batch(batch_size)

train_ds = df_to_dataset(X_train, y_train)
val_ds = df_to_dataset(X_val, y_val)
test_ds = df_to_dataset(X_test, y_test, shuffle=False)

# Construir el modelo
def build_model(layer_units, input_features):
    inputs = [keras.Input(shape=(1,), name=col) for col in input_features]
    x = keras.layers.concatenate(inputs)
    
    for units in layer_units:
        x = keras.layers.Dense(units, activation='relu')(x)
    
    output = keras.layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Configurar MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("redesNeuronales-clas-icfes")

# Parámetros a probar
# arquitectura = [64, 32, 16]
# 3 capas con 64, 32 y 16 neuronas
arquitectura = [64, 32]

with mlflow.start_run():
    mlflow.log_param("arquitectura", arquitectura)

    model = build_model(arquitectura, X_train.columns)
    mlflow.tensorflow.autolog()

    model.fit(train_ds, validation_data=val_ds, epochs=50)

    loss, accuracy = model.evaluate(test_ds)
    mlflow.log_metric("test_loss", loss)
    mlflow.log_metric("test_accuracy", accuracy)

    model.save("modelo_binarioRN2.h5")
    mlflow.log_artifact("modelo_binarioRN2.h5")

    y_pred = model.predict(test_ds)
    y_pred_class = (y_pred > 0.5).astype("int")

    conf_matrix = confusion_matrix(y_test, y_pred_class)
    classif_report = classification_report(y_test, y_pred_class, output_dict=True)

    mlflow.log_dict(classif_report, "classification_report.json")


2025/05/25 19:21:08 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.
2025/05/25 19:21:39 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.


Epoch 1/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7030 - loss: 0.5629

9528/9528 ━━━━━━━━━━━━━━━━━━━━ 166s 16ms/step - accuracy: 0.7030 - loss: 0.5629 - val_accuracy: 0.7095 - val_loss: 0.5522
Epoch 2/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7107 - loss: 0.5509

9528/9528 ━━━━━━━━━━━━━━━━━━━━ 258s 19ms/step - accuracy: 0.7107 - loss: 0.5509 - val_accuracy: 0.7097 - val_loss: 0.5522
Epoch 3/50
9527/9528 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7125 - loss: 0.5486

9528/9528 ━━━━━━━━━━━━━━━━━━━━ 273s 20ms/step - accuracy: 0.7125 - loss: 0.5486 - val_accuracy: 0.7104 - val_loss: 0.5501
Epoch 4/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7124 - loss: 0.5477

9528/9528 ━━━━━━━━━━━━━━━━━━━━ 231s 20ms/step - accuracy: 0.7124 - loss: 0.5477 - val_accuracy: 0.7112 - val_loss: 0.5498
Epoch 5/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 214s 17ms/step - accuracy: 0.7147 - loss: 0.5451 - val_accuracy: 0.7106 - val_loss: 0.5508
Epoch 6/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 76s 6ms/step - accuracy: 0.7149 - loss: 0.5451 - val_accuracy: 0.7110 - val_loss: 0.5508
Epoch 7/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 104s 9ms/step - accuracy: 0.7144 - loss: 0.5442 - val_accuracy: 0.7108 - val_loss: 0.5500
Epoch 8/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 133s 11ms/step - accuracy: 0.7170 - loss: 0.5418 - val_accuracy: 0.7110 - val_loss: 0.5502
Epoch 9/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 159s 11ms/step - accuracy: 0.7176 - loss: 0.5422 - val_accuracy: 0.7102 - val_loss: 0.5508
Epoch 10/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 94s 7ms/step - accuracy: 0.7165 - loss: 0.5415 - val_accuracy: 0.7114 - val_loss: 0.5511
Epoch 11/50
9528/9528 ━━━━━━━━━━━━━━━━━━━━ 87s 7ms/step - accuracy: 0.7183 - loss: 0

🏃 View run delightful-loon-662 at: http://localhost:5000/#/experiments/392861474054559561/runs/0d894532d3344b2492f30890ba3ae964
🧪 View experiment at: http://localhost:5000/#/experiments/392861474054559561


FileNotFoundError: [Errno 2] No such file or directory: 'modelo_RN2.h5'

In [4]:
model.save("modelo_binarioRN2.h5")
mlflow.log_artifact("modelo_binarioRN2.h5")